# Stage 1.5 -- Validation & Feature Engineering: Panel D (Macro Monthly)

## Input
`Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_macro_monthly.parquet` (~252 rows, ~107 columns, keyed on `date` only)

## Purpose
Validation and feature engineering of the merged macro monthly panel. Panel D contains publication-lag-adjusted monthly macro indicators (FRED with 2-month lag, WRDS Treasury/liquidity with no lag, WRDS CPI with 1-month lag). The panel is market-level with no cross-sectional dimension. Feature engineering is deliberately minimal — the FRED cleaning stage already computed 24 MoM changes and 5 YoY changes, so only features that genuinely add new information are created: economic acceleration (2nd derivative of MoM), bond term premium return spreads, and smoothed MoM averages.

---

## Initial Diagnostics

A diagnostic pass (run before Block 1) lists all factor columns with NaN rates, min/max ranges, checks for non-numeric columns, infinite values, and tests all factor pairs for exact duplicate columns. NaN structure is analysed by date to identify which months have missing data (expected: early months due to publication lag shifting). No exact duplicates are found.

---

## Block 1: Clean & Diagnose

### Step 1: Replace Infinite Values with NaN
Safety net scan across all numeric columns. Any ±inf values are replaced with NaN.

### Step 2: Near-Zero Variance Check
Scans all factor columns for effectively constant values (std < 1e-8).

### Step 3: Structural NaN Analysis
Reports all columns with any NaN, by month and by column, showing NaN count, percentage, and first valid date. Identifies:
- **FRED factors (2-month lag):** January and February 2004 have NaN for all FRED factors (reference data shifted forward by 2 months)
- **WRDS CPI (1-month lag):** January 2004 has NaN
- **`avg_hourly_earnings`:** structural late start (BLS series begins March 2006)

### Step 4: Date Range Trim Decision
Two options are evaluated programmatically:
- **Option A:** drop January--February 2004 only (preserves 250 months)
- **Option B:** align with Panel C post-June 2006 (preserves 222 months, guaranteed zero NaN including `avg_hourly_earnings`)

Option B is executed: data trimmed to post-June 2006. Zero NaN confirmed across all factors after trim.

---

## Block 2: Complete Factor Inventory (Pre-Engineering)

Every surviving factor catalogued with column name, source, category, and description. Organised into themed groups:

- **Labor Market (8):** nonfarm payrolls, unemployment rate, U-6 underemployment, participation rate, average hourly earnings, average weekly hours, JOLTS openings, JOLTS quits rate
- **Inflation & Prices (9):** CPI-U all items, core CPI, food, energy, shelter, services, PCE price index, core PCE, import prices
- **Production (3):** industrial production, capacity utilisation, manufacturing production
- **Orders & Inventories (5):** durable goods orders, durables ex-transport, factory orders, business inventories, inventory-to-sales ratio
- **Consumer (8):** retail sales, retail ex-auto, personal income, personal spending, saving rate, consumer credit, Michigan sentiment, vehicle sales
- **Housing (4):** housing starts, building permits, new home sales, Case-Shiller HPI
- **Trade (4):** trade balance, exports, imports, trade balance 12-month rolling average
- **Money Supply (3):** M1, M2, monetary base
- **Commodities (5):** copper, aluminum, wheat, corn, lumber (monthly averages)
- **Surveys (6):** ISM manufacturing PMI, ISM new orders, Philly Fed, Empire State, Kansas City Fed, Chicago Fed National Activity Index
- **MoM Changes (24):** pre-computed in FRED cleaning for payrolls, CPI, PCE, industrial production, retail sales, personal income/spending, housing starts, permits, new home sales, durables, factory orders, exports, imports, M1/M2/monetary base, copper, Case-Shiller, consumer credit
- **YoY Changes (5):** CPI-U, core CPI, PCE, core PCE, Case-Shiller
- **WRDS Treasury Bond Returns & Indices (14):** monthly returns and total return indices for 1Y, 2Y, 5Y, 7Y, 10Y, 20Y, 30Y maturities
- **WRDS T-Bill Returns & Indices (4):** 30-day and 90-day T-bill returns and indices
- **WRDS Pastor-Stambaugh Liquidity (2):** level and innovation
- **WRDS CPI (2):** monthly CPI return and index level

Inventory validated bidirectionally and saved as `macro_monthly_descriptions_pre.csv`.

---

## Block 3: Feature Engineering

Three sections creating 27 new features. No columns are dropped.

### Section A: Economic Acceleration -- 2nd Derivative of MoM (11 features)

Acceleration is computed as the **first difference of the MoM change** (`df[src].diff(1)`), i.e. this month's MoM minus last month's MoM. Positive = the rate of change is increasing (economy accelerating), negative = decelerating. Applied to 11 key series:

`nfp_accel`, `cpi_core_accel`, `pce_core_accel`, `retail_accel`, `indpro_accel`, `housing_starts_accel`, `durable_orders_accel`, `spending_accel`, `income_accel`, `exports_accel`, `imports_accel`

### Section B: Bond Term Premium Returns (6 features)

Return spreads between long and short duration bonds, capturing duration risk compensation and curve dynamics. Positive = long bonds outperformed (rates fell or curve steepened):

- `bond_30y_2y_spread_ret` = b30ret - b2ret (long-duration vs short-duration)
- `bond_10y_tbill_spread_ret` = b10ret - t90ret (benchmark term premium)
- `bond_30y_10y_spread_ret` = b30ret - b10ret (long-end steepener/flattener)
- `bond_5y_2y_spread_ret` = b5ret - b2ret (belly of the curve)
- `b10ret_cum_3m` = 3-month rolling sum of 10Y bond return (bond momentum)
- `b30ret_cum_3m` = 3-month rolling sum of 30Y bond return (bond momentum)

### Section C: Smoothed MoM -- 3-Month Rolling Averages (10 features)

3-month rolling means (`rolling(3, min_periods=2).mean()`) of the most volatile monthly series, giving a cleaner picture of the underlying trend vs single-month noise. Applied to: nonfarm payrolls MoM, retail sales MoM, housing starts MoM, durable orders MoM, industrial production MoM, personal spending MoM, new home sales MoM, core CPI MoM, factory orders MoM, exports MoM.

---

## Block 4: Final Factor Inventory & Save

Every surviving factor catalogued in a final inventory DataFrame with column name, source, category, and description. Validated bidirectionally against the actual data columns.

**Final factor count: 133** (107 original factors + 27 new engineered features, 0 dropped)

**Factor breakdown by section:**
- FRED original levels and derived (MoM/YoY): ~106 factors
- WRDS bond returns/indices, T-bills, PS liquidity, CPI: ~22 factors
- Derived (acceleration, bond term premium, smoothed MoM): 27 factors

**Key properties after engineering:**
- 222 rows (July 2006 -- December 2024), zero NaN
- Publication lags pre-applied from Stage 1 merge
- No cross-sectional dimension -- passes through directly to Stage 2 without aggregation

## Outputs
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_monthly_engineered.parquet` -- 133 factor columns plus `date`, zero NaN, post-June 2006
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/macro_monthly_descriptions_pre.csv` -- pre-engineering factor inventory
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/macro_monthly_factor_inventory_final.csv` -- final post-engineering factor inventory with source, category, description

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PANEL_D_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_macro_monthly.parquet')

df = pd.read_parquet(PANEL_D_PATH)
df['date'] = pd.to_datetime(df['date'])

factor_cols = [c for c in df.columns if c != 'date']

print(f"Panel D: {len(df):,} rows × {df.shape[1]} columns")
print(f"Factors: {len(factor_cols)}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Rows per year: {len(df) / df['date'].dt.year.nunique():.1f}")

# Non-numeric
non_numeric = [c for c in factor_cols if not pd.api.types.is_numeric_dtype(df[c])]
print(f"Non-numeric: {non_numeric}")

# Infinites
print(f"\nInfinite values:")
inf_found = False
for c in factor_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        n_inf = np.isinf(df[c]).sum()
        if n_inf > 0:
            print(f"  {c}: {n_inf}")
            inf_found = True
if not inf_found:
    print(f"  ✓ None")

# Full column list
print(f"\n{'#':<5} {'Column':<40} {'NaN%':>7}  {'NaN#':>5}  {'Min':>14}  {'Max':>14}")
print("-" * 90)
for i, c in enumerate(factor_cols, 1):
    nan_p = df[c].isna().mean() * 100
    nan_n = df[c].isna().sum()
    if pd.api.types.is_numeric_dtype(df[c]):
        vals = df[c].dropna()
        if len(vals) > 0:
            try:
                print(f"{i:<5} {c:<40} {nan_p:>6.2f}%  {nan_n:>5d}  {float(vals.min()):>14.4f}  {float(vals.max()):>14.4f}")
            except:
                print(f"{i:<5} {c:<40} {nan_p:>6.2f}%  {nan_n:>5d}  {'error':>14}  {'error':>14}")
        else:
            print(f"{i:<5} {c:<40} {nan_p:>6.2f}%  {nan_n:>5d}  {'all NaN':>14}  {'':>14}")
    else:
        print(f"{i:<5} {c:<40} {nan_p:>6.2f}%  {nan_n:>5d}  {'non-numeric':>14}  {'':>14}")

# Exact duplicate check
print(f"\n--- Exact duplicate check (all rows) ---")
exact_dupes = []
for i, c1 in enumerate(factor_cols):
    for c2 in factor_cols[i+1:]:
        if pd.api.types.is_numeric_dtype(df[c1]) and pd.api.types.is_numeric_dtype(df[c2]):
            if df[c1].equals(df[c2]):
                exact_dupes.append((c1, c2))
if exact_dupes:
    for c1, c2 in exact_dupes:
        print(f"  {c1} == {c2}")
else:
    print(f"  ✓ No exact duplicates")

# NaN structure: which months have NaN?
print(f"\n--- NaN by date (first 5 months with any NaN) ---")
nan_by_date = df.set_index('date')[factor_cols].isna().sum(axis=1)
nan_dates = nan_by_date[nan_by_date > 0]
if len(nan_dates) > 0:
    for dt, n in nan_dates.head(5).items():
        print(f"  {dt.date()}: {n} factors with NaN")
    print(f"  ... total dates with any NaN: {len(nan_dates)}")
else:
    print(f"  ✓ No NaN in any month")

Panel D: 252 rows × 107 columns
Factors: 106
Date range: 2004-01-31 → 2024-12-31
Rows per year: 12.0
Non-numeric: []

Infinite values:
  ✓ None

#     Column                                      NaN%   NaN#             Min             Max
------------------------------------------------------------------------------------------
1     nonfarm_payrolls                           0.79%      2     129526.0000     159105.0000
2     unrate                                     0.79%      2          3.4000         14.7000
3     u6_rate                                    0.79%      2          6.5000         22.8000
4     participation                              0.79%      2         60.1000         66.4000
5     avg_hourly_earnings                       11.11%     28         20.0500         35.4600
6     avg_weekly_hours                           0.79%      2         38.7000         42.4000
7     jolts_openings                             0.79%      2       2338.0000      11549.0000
8     jolts_

In [2]:
# %% [markdown]
# # Stage 1.5 — Validation & Feature Engineering: Panel D (Macro Monthly)
#
# Panel D is market-level monthly macro data. 252 months × 106 factors.
# Simplest panel — mostly already ratios/indices, minimal engineering needed.
#
# Block 1: Clean infinites, diagnose NaN, handle gaps
# Block 2: Inventory of surviving factors
# Block 3: Feature engineering (minimal — mostly documentation)
# Block 4: Final inventory & save
#
# Input:  Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_macro_monthly.parquet
# Output: Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_monthly_engineered.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path

IN_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_macro_monthly.parquet')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(IN_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
n_start = df.shape[1]

factor_cols = [c for c in df.columns if c != 'date']

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 1: CLEAN & DIAGNOSE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("BLOCK 1: CLEAN & DIAGNOSE")
print("=" * 90)

# ── Step 1: Replace ±inf with NaN ────────────────────────────────────────────
print("\n--- Step 1: Replace ±inf with NaN ---")

numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_count = 0
for c in numeric_cols:
    n_inf = np.isinf(df[c]).sum()
    if n_inf > 0:
        print(f"  {c}: {n_inf} inf")
        inf_count += n_inf

df = df.replace([np.inf, -np.inf], np.nan)

if inf_count == 0:
    print(f"  ✓ No infinite values found")
else:
    print(f"  Replaced {inf_count} infinite values with NaN")

# ── Step 2: Near-zero variance ──────────────────────────────────────────────
print("\n--- Step 2: Near-zero variance check ---")

low_var = []
for c in factor_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        vals = df[c].dropna()
        if len(vals) > 10:
            std = vals.std()
            if pd.notna(std) and float(std) < 1e-8:
                low_var.append(c)
                print(f"  {c}: std={float(std):.2e}")

if not low_var:
    print(f"  ✓ No near-zero variance factors")

# ── Step 3: NaN structure analysis ──────────────────────────────────────────
print("\n--- Step 3: NaN structure analysis ---")

# 3a. NaN by month
print(f"\n  NaN by month (showing months with >0 NaN):")
print(f"  {'Date':<12s} {'NaN cols':>9s}  {'Example columns'}")
print("  " + "-" * 70)
for _, row in df.iterrows():
    nan_cols_here = [c for c in factor_cols if pd.isna(row[c])]
    if nan_cols_here:
        examples = ', '.join(nan_cols_here[:4])
        if len(nan_cols_here) > 4:
            examples += f', ... (+{len(nan_cols_here) - 4} more)'
        print(f"  {row['date'].date()!s:<12s} {len(nan_cols_here):>9d}  {examples}")

# 3b. NaN by column
print(f"\n  Columns with any NaN:")
print(f"  {'Column':<40s} {'NaN':>5s}  {'%':>6s}  {'First Valid':>12s}")
print("  " + "-" * 70)
for c in factor_cols:
    n = df[c].isna().sum()
    if n > 0:
        pct = n / len(df) * 100
        first_valid = df[df[c].notna()]['date'].min()
        fv = first_valid.strftime('%Y-%m-%d') if pd.notna(first_valid) else 'never'
        print(f"  {c:<40s} {n:>5d}  {pct:>5.2f}%  {fv:>12s}")

# 3c. Summary: if we drop Jan-Feb 2004, how much NaN remains?
print(f"\n  Impact of dropping Jan-Feb 2004:")
post_feb = df[df['date'] >= '2004-03-01']
post_feb_nan = post_feb[factor_cols].isna().sum()
remaining_nan_cols = post_feb_nan[post_feb_nan > 0]
print(f"    Rows: {len(df)} → {len(post_feb)}")
if len(remaining_nan_cols) > 0:
    print(f"    Columns still with NaN: {len(remaining_nan_cols)}")
    for c in remaining_nan_cols.index:
        print(f"      {c}: {int(remaining_nan_cols[c])} NaN")
else:
    print(f"    ✓ Zero NaN remaining")

# 3d. Check what happens if we align with Panel C (post June 2006)
print(f"\n  Impact of aligning with Panel C (post June 2006):")
post_june06 = df[df['date'] >= '2006-07-01']
post_june06_nan = post_june06[factor_cols].isna().sum()
remaining_nan_post06 = post_june06_nan[post_june06_nan > 0]
print(f"    Rows: {len(df)} → {len(post_june06)} ({len(df) - len(post_june06)} months lost)")
if len(remaining_nan_post06) > 0:
    print(f"    Columns still with NaN: {len(remaining_nan_post06)}")
    for c in remaining_nan_post06.index:
        print(f"      {c}: {int(remaining_nan_post06[c])} NaN")
else:
    print(f"    ✓ Zero NaN remaining")

# ── Step 4: Decision and execution ──────────────────────────────────────────
print("\n--- Step 4: Decision ---")

# Check: does dropping just Jan-Feb 2004 solve the problem?
# Or do we need to drop more / handle avg_hourly_earnings separately?

# Count total NaN before and after potential fixes
total_nan_before = df[factor_cols].isna().sum().sum()
print(f"\n  Total NaN (all data): {total_nan_before}")

# Option A: drop first 2 months only
option_a = df[df['date'] >= '2004-03-01'].copy()
nan_a = option_a[factor_cols].isna().sum().sum()
print(f"  Option A (drop Jan-Feb 2004): {nan_a} NaN remaining, {len(option_a)} months")

# Option B: align with Panel C (post June 2006)  
option_b = df[df['date'] >= '2006-07-01'].copy()
nan_b = option_b[factor_cols].isna().sum().sum()
print(f"  Option B (align with Panel C, post Jun 2006): {nan_b} NaN remaining, {len(option_b)} months")

# Execute based on results
# (Print recommendation — user decides)
print(f"\n  Recommendation:")
if nan_a == 0:
    print(f"  → Option A: drop Jan-Feb 2004 only (preserves 250 months, zero NaN)")
elif nan_b == 0:
    print(f"  → Option B needed: some NaN persists after dropping Jan-Feb 2004")
    print(f"    avg_hourly_earnings has structural NaN — consider Option B or ffill")
else:
    print(f"  → Neither option eliminates all NaN — investigate remaining columns")

# ── For now: print what we'd have with each option ──────────────────────────
print(f"\n  Awaiting user decision before dropping rows.")
print(f"  Run this cell to execute your choice:")
print(f"""
  # OPTION A: drop Jan-Feb 2004 only
  # df = df[df['date'] >= '2004-03-01'].reset_index(drop=True)
  
  # OPTION B: align with Panel C post June 2006
  # df = df[df['date'] >= '2006-07-01'].reset_index(drop=True)
""")

# Final state
remaining = [c for c in df.columns if c != 'date']
print(f"\n  Current state: {len(df):,} rows × {df.shape[1]} columns")
print(f"  Factors: {len(remaining)}")
print(f"  ✓ Date column intact")

Loaded: 252 rows × 107 columns
BLOCK 1: CLEAN & DIAGNOSE

--- Step 1: Replace ±inf with NaN ---
  ✓ No infinite values found

--- Step 2: Near-zero variance check ---
  ✓ No near-zero variance factors

--- Step 3: NaN structure analysis ---

  NaN by month (showing months with >0 NaN):
  Date          NaN cols  Example columns
  ----------------------------------------------------------------------
  2004-01-31          86  nonfarm_payrolls, unrate, u6_rate, participation, ... (+82 more)
  2004-02-29          84  nonfarm_payrolls, unrate, u6_rate, participation, ... (+80 more)
  2004-03-31           1  avg_hourly_earnings
  2004-04-30           1  avg_hourly_earnings
  2004-05-31           1  avg_hourly_earnings
  2004-06-30           1  avg_hourly_earnings
  2004-07-31           1  avg_hourly_earnings
  2004-08-31           1  avg_hourly_earnings
  2004-09-30           1  avg_hourly_earnings
  2004-10-31           1  avg_hourly_earnings
  2004-11-30           1  avg_hourly_earnings
  

In [3]:
# NO DATE TRIM. Panel D keeps 2004-01-31 onward.
#
# The Step 3 diagnostic above shows Option A (drop Jan-Feb 2004) leaves NaN
# behind, and names avg_hourly_earnings as the reason -- FRED CES0500000003
# genuinely begins March 2006, so with the 2-month publication lag it enters at
# May 2006. That is a late series, not a defect, and per-feature warm-up in
# Stage 3 is what it calls for. Trimming to 2006-07 threw away 30 months of
# history for 105 other factors to accommodate one.

factor_cols = [c for c in df.columns if c != 'date']

print(f"Rows: {len(df)}, Factors: {len(factor_cols)}")
print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")

nan_by_col = df[factor_cols].isna().sum()
nan_by_col = nan_by_col[nan_by_col > 0].sort_values(ascending=False)
print(f"\nFactors with any NaN: {len(nan_by_col)} / {len(factor_cols)}")
for c in nan_by_col.index:
    fv = df.loc[df[c].notna(), 'date'].min()
    print(f"  {c:<32s} {int(nan_by_col[c]):>4d} NaN   first valid {fv.date()}")
print("\nExpected: FRED NaN for Jan-Feb 2004 (2-month lag), CPI for Jan 2004")
print("(1-month lag), avg_hourly_earnings until May 2006 (series starts Mar 2006).")

Rows: 252, Factors: 106
Date range: 2004-01-31 -> 2024-12-31

Factors with any NaN: 86 / 106
  avg_hourly_earnings                28 NaN   first valid 2006-05-31
  nonfarm_payrolls                    2 NaN   first valid 2004-03-31
  nonfarm_payrolls_mom                2 NaN   first valid 2004-03-31
  personal_income_mom                 2 NaN   first valid 2004-03-31
  retail_ex_auto_mom                  2 NaN   first valid 2004-03-31
  retail_sales_mom                    2 NaN   first valid 2004-03-31
  indpro_mom                          2 NaN   first valid 2004-03-31
  pce_core_mom                        2 NaN   first valid 2004-03-31
  pce_price_mom                       2 NaN   first valid 2004-03-31
  cpi_core_mom                        2 NaN   first valid 2004-03-31
  cpi_urban_mom                       2 NaN   first valid 2004-03-31
  chicago_fed_nai                     2 NaN   first valid 2004-03-31
  housing_starts_mom                  2 NaN   first valid 2004-03-31
  kansas_f

In [4]:
# %% [markdown]
# ## Block 2: Complete Factor Inventory
#
# Every surviving factor catalogued with source, category, and description.

# %%
print("=" * 90)
print("BLOCK 2: COMPLETE FACTOR INVENTORY")
print("=" * 90)

inventory = []

def add(col, source, category, description):
    inventory.append({
        'column': col,
        'source': source,
        'category': category,
        'description': description,
    })

# ─────────────────────────────────────────────────────────────────────────────
# LABOR MARKET — LEVELS (8)
# ─────────────────────────────────────────────────────────────────────────────

add('nonfarm_payrolls',    'FRED', 'labor_market', 'Total nonfarm payrolls (thousands of persons)')
add('unrate',              'FRED', 'labor_market', 'Unemployment rate (%)')
add('u6_rate',             'FRED', 'labor_market', 'U-6 underemployment rate (%)')
add('participation',       'FRED', 'labor_market', 'Labor force participation rate (%)')
add('avg_hourly_earnings', 'FRED', 'labor_market', 'Average hourly earnings, private sector ($)')
add('avg_weekly_hours',    'FRED', 'labor_market', 'Average weekly hours, private sector')
add('jolts_openings',      'FRED', 'labor_market', 'JOLTS job openings (thousands)')
add('jolts_quits',         'FRED', 'labor_market', 'JOLTS quits rate (%)')

# ─────────────────────────────────────────────────────────────────────────────
# INFLATION & PRICES — LEVELS (9)
# ─────────────────────────────────────────────────────────────────────────────

add('cpi_urban',     'FRED', 'inflation', 'CPI-U all items (index level)')
add('cpi_core',      'FRED', 'inflation', 'CPI-U less food & energy (index level)')
add('cpi_food',      'FRED', 'inflation', 'CPI food component (index level)')
add('cpi_energy',    'FRED', 'inflation', 'CPI energy component (index level)')
add('cpi_shelter',   'FRED', 'inflation', 'CPI shelter component (index level)')
add('cpi_services',  'FRED', 'inflation', 'CPI services component (index level)')
add('pce_price',     'FRED', 'inflation', 'PCE price index (index level)')
add('pce_core',      'FRED', 'inflation', 'Core PCE price index (index level)')
add('import_prices', 'FRED', 'inflation', 'Import price index (index level)')

# ─────────────────────────────────────────────────────────────────────────────
# INDUSTRIAL PRODUCTION (3)
# ─────────────────────────────────────────────────────────────────────────────

add('indpro',     'FRED', 'production', 'Industrial production index')
add('cap_util',   'FRED', 'production', 'Capacity utilisation rate (%)')
add('manuf_prod', 'FRED', 'production', 'Manufacturing production index')

# ─────────────────────────────────────────────────────────────────────────────
# ORDERS & INVENTORIES (5)
# ─────────────────────────────────────────────────────────────────────────────

add('durable_orders',       'FRED', 'orders', 'Durable goods orders (millions $)')
add('durable_ex_transport', 'FRED', 'orders', 'Durable goods ex-transportation (millions $)')
add('factory_orders',       'FRED', 'orders', 'Factory orders (millions $)')
add('business_inventories', 'FRED', 'orders', 'Business inventories (millions $)')
add('inv_sales_ratio',      'FRED', 'orders', 'Inventory-to-sales ratio')

# ─────────────────────────────────────────────────────────────────────────────
# CONSUMER (8)
# ─────────────────────────────────────────────────────────────────────────────

add('retail_sales',      'FRED', 'consumer', 'Retail sales (millions $)')
add('retail_ex_auto',    'FRED', 'consumer', 'Retail sales ex-auto (millions $)')
add('personal_income',   'FRED', 'consumer', 'Personal income (billions $)')
add('personal_spending', 'FRED', 'consumer', 'Personal consumption expenditure (billions $)')
add('saving_rate',       'FRED', 'consumer', 'Personal saving rate (%)')
add('consumer_credit',   'FRED', 'consumer', 'Consumer credit outstanding (billions $)')
add('umich_sentiment',   'FRED', 'consumer', 'University of Michigan consumer sentiment index')
add('vehicle_sales',     'FRED', 'consumer', 'Total vehicle sales (millions of units, SAAR)')

# ─────────────────────────────────────────────────────────────────────────────
# HOUSING (4)
# ─────────────────────────────────────────────────────────────────────────────

add('housing_starts',   'FRED', 'housing', 'Housing starts (thousands of units, SAAR)')
add('building_permits', 'FRED', 'housing', 'Building permits (thousands of units, SAAR)')
add('new_home_sales',   'FRED', 'housing', 'New single-family home sales (thousands, SAAR)')
add('case_shiller',     'FRED', 'housing', 'S&P/Case-Shiller national home price index')

# ─────────────────────────────────────────────────────────────────────────────
# TRADE (3)
# ─────────────────────────────────────────────────────────────────────────────

add('trade_balance', 'FRED', 'trade', 'Trade balance: exports - imports (millions $, negative = deficit)')
add('exports',       'FRED', 'trade', 'Total exports (millions $)')
add('imports',       'FRED', 'trade', 'Total imports (millions $)')

# ─────────────────────────────────────────────────────────────────────────────
# MONEY SUPPLY (3)
# ─────────────────────────────────────────────────────────────────────────────

add('m1',            'FRED', 'money_supply', 'M1 money supply (billions $)')
add('m2',            'FRED', 'money_supply', 'M2 money supply (billions $)')
add('monetary_base', 'FRED', 'money_supply', 'Monetary base (billions $)')

# ─────────────────────────────────────────────────────────────────────────────
# COMMODITIES — MONTHLY (5)
# ─────────────────────────────────────────────────────────────────────────────

add('copper_monthly',   'FRED', 'commodities', 'Copper price monthly avg ($/metric ton)')
add('aluminum_monthly', 'FRED', 'commodities', 'Aluminum price monthly avg ($/metric ton)')
add('wheat_monthly',    'FRED', 'commodities', 'Wheat price monthly avg ($/bushel)')
add('corn_monthly',     'FRED', 'commodities', 'Corn price monthly avg ($/bushel)')
add('lumber_monthly',   'FRED', 'commodities', 'Lumber price monthly avg ($/1000 board ft)')

# ─────────────────────────────────────────────────────────────────────────────
# SURVEYS & REGIONAL FED (6)
# ─────────────────────────────────────────────────────────────────────────────

add('ism_manuf',       'FRED', 'surveys', 'ISM Manufacturing PMI (index, 50 = expansion/contraction)')
add('ism_new_orders',  'FRED', 'surveys', 'ISM Manufacturing new orders (index)')
add('philly_fed',      'FRED', 'surveys', 'Philadelphia Fed Manufacturing index')
add('empire_state',    'FRED', 'surveys', 'Empire State Manufacturing index')
add('kansas_fed',      'FRED', 'surveys', 'Kansas City Fed Manufacturing index')
add('chicago_fed_nai', 'FRED', 'surveys', 'Chicago Fed National Activity Index')

# ─────────────────────────────────────────────────────────────────────────────
# MONTH-OVER-MONTH CHANGES (24) — computed in FRED cleaning
# ─────────────────────────────────────────────────────────────────────────────

add('nonfarm_payrolls_mom',    'FRED', 'labor_market_mom',  'Nonfarm payrolls MoM % change')
add('cpi_urban_mom',           'FRED', 'inflation_mom',     'CPI-U all items MoM % change')
add('cpi_core_mom',            'FRED', 'inflation_mom',     'Core CPI MoM % change')
add('pce_price_mom',           'FRED', 'inflation_mom',     'PCE price index MoM % change')
add('pce_core_mom',            'FRED', 'inflation_mom',     'Core PCE MoM % change')
add('indpro_mom',              'FRED', 'production_mom',    'Industrial production MoM % change')
add('retail_sales_mom',        'FRED', 'consumer_mom',      'Retail sales MoM % change')
add('retail_ex_auto_mom',      'FRED', 'consumer_mom',      'Retail sales ex-auto MoM % change')
add('personal_income_mom',     'FRED', 'consumer_mom',      'Personal income MoM % change')
add('personal_spending_mom',   'FRED', 'consumer_mom',      'Personal spending MoM % change')
add('housing_starts_mom',      'FRED', 'housing_mom',       'Housing starts MoM % change')
add('building_permits_mom',    'FRED', 'housing_mom',       'Building permits MoM % change')
add('new_home_sales_mom',      'FRED', 'housing_mom',       'New home sales MoM % change')
add('durable_orders_mom',      'FRED', 'orders_mom',        'Durable goods orders MoM % change')
add('durable_ex_transport_mom','FRED', 'orders_mom',        'Durables ex-transport MoM % change')
add('factory_orders_mom',      'FRED', 'orders_mom',        'Factory orders MoM % change')
add('exports_mom',             'FRED', 'trade_mom',         'Exports MoM % change')
add('imports_mom',             'FRED', 'trade_mom',         'Imports MoM % change')
add('m1_mom',                  'FRED', 'money_supply_mom',  'M1 MoM % change')
add('m2_mom',                  'FRED', 'money_supply_mom',  'M2 MoM % change')
add('monetary_base_mom',       'FRED', 'money_supply_mom',  'Monetary base MoM % change')
add('copper_monthly_mom',      'FRED', 'commodities_mom',   'Copper price MoM % change')
add('case_shiller_mom',        'FRED', 'housing_mom',       'Case-Shiller home price MoM % change')
add('consumer_credit_mom',     'FRED', 'consumer_mom',      'Consumer credit MoM % change')

# ─────────────────────────────────────────────────────────────────────────────
# YEAR-OVER-YEAR CHANGES (5) — computed in FRED cleaning
# ─────────────────────────────────────────────────────────────────────────────

add('cpi_urban_yoy',    'FRED', 'inflation_yoy', 'CPI-U all items YoY % change (headline inflation)')
add('cpi_core_yoy',     'FRED', 'inflation_yoy', 'Core CPI YoY % change')
add('pce_price_yoy',    'FRED', 'inflation_yoy', 'PCE price index YoY % change')
add('pce_core_yoy',     'FRED', 'inflation_yoy', 'Core PCE YoY % change (Fed preferred measure)')
add('case_shiller_yoy', 'FRED', 'housing_yoy',   'Case-Shiller home price YoY % change')

# ─────────────────────────────────────────────────────────────────────────────
# DERIVED TRADE (1)
# ─────────────────────────────────────────────────────────────────────────────

add('trade_balance_12m_avg', 'FRED', 'trade', 'Trade balance 12-month rolling average (millions $)')

# ─────────────────────────────────────────────────────────────────────────────
# WRDS TREASURY BOND RETURNS & INDICES (14)
# ─────────────────────────────────────────────────────────────────────────────

add('b30ret', 'WRDS', 'bond_returns', '30-year Treasury bond monthly return')
add('b30ind', 'WRDS', 'bond_returns', '30-year Treasury bond total return index')
add('b20ret', 'WRDS', 'bond_returns', '20-year Treasury bond monthly return')
add('b20ind', 'WRDS', 'bond_returns', '20-year Treasury bond total return index')
add('b10ret', 'WRDS', 'bond_returns', '10-year Treasury bond monthly return')
add('b10ind', 'WRDS', 'bond_returns', '10-year Treasury bond total return index')
add('b7ret',  'WRDS', 'bond_returns', '7-year Treasury bond monthly return')
add('b7ind',  'WRDS', 'bond_returns', '7-year Treasury bond total return index')
add('b5ret',  'WRDS', 'bond_returns', '5-year Treasury bond monthly return')
add('b5ind',  'WRDS', 'bond_returns', '5-year Treasury bond total return index')
add('b2ret',  'WRDS', 'bond_returns', '2-year Treasury bond monthly return')
add('b2ind',  'WRDS', 'bond_returns', '2-year Treasury bond total return index')
add('b1ret',  'WRDS', 'bond_returns', '1-year Treasury bond monthly return')
add('b1ind',  'WRDS', 'bond_returns', '1-year Treasury bond total return index')

# ─────────────────────────────────────────────────────────────────────────────
# WRDS T-BILL RETURNS & INDICES (4)
# ─────────────────────────────────────────────────────────────────────────────

add('t90ret', 'WRDS', 'tbill_returns', '90-day T-bill monthly return')
add('t90ind', 'WRDS', 'tbill_returns', '90-day T-bill total return index')
add('t30ret', 'WRDS', 'tbill_returns', '30-day T-bill monthly return')
add('t30ind', 'WRDS', 'tbill_returns', '30-day T-bill total return index')

# ─────────────────────────────────────────────────────────────────────────────
# WRDS PASTOR-STAMBAUGH LIQUIDITY (2)
# ─────────────────────────────────────────────────────────────────────────────

add('ps_level', 'WRDS', 'liquidity', 'Pastor-Stambaugh aggregate liquidity level')
add('ps_innov', 'WRDS', 'liquidity', 'Pastor-Stambaugh aggregate liquidity innovation')

# ─────────────────────────────────────────────────────────────────────────────
# WRDS CPI (2)
# ─────────────────────────────────────────────────────────────────────────────

add('cpiret', 'WRDS', 'inflation', 'Monthly CPI return (from WRDS)')
add('cpiind', 'WRDS', 'inflation', 'CPI index level (from WRDS)')

# ═══════════════════════════════════════════════════════════════════════════════
# VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
inv = pd.DataFrame(inventory)

remaining_factors = [c for c in df.columns if c != 'date']
catalogued = set(inv['column'])

in_data_not_catalogued = [c for c in remaining_factors if c not in catalogued]
in_catalogue_not_data = [c for c in catalogued if c not in remaining_factors]

print(f"\n  Factors in data:       {len(remaining_factors)}")
print(f"  Factors catalogued:    {len(inv)}")

if in_data_not_catalogued:
    print(f"\n  ✗ IN DATA but NOT catalogued ({len(in_data_not_catalogued)}):")
    for c in in_data_not_catalogued:
        print(f"    {c}")
else:
    print(f"  ✓ Every factor in data is catalogued")

if in_catalogue_not_data:
    print(f"\n  ✗ In catalogue but NOT in data ({len(in_catalogue_not_data)}):")
    for c in in_catalogue_not_data:
        print(f"    {c}")
else:
    print(f"  ✓ Every catalogued factor exists in data")

# Summaries
print(f"\n  By source:")
print(inv['source'].value_counts().to_string())

print(f"\n  By category:")
print(inv['category'].value_counts().to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
csv_path = OUT_DIR / 'macro_monthly_descriptions_pre.csv'
csv_string = inv.to_csv(index=False)
with open(csv_path, 'w', encoding='utf-8') as f:
    f.write(csv_string)
print(f"\n  ✓ Saved: {csv_path}")
print(f"    {len(inv)} factors")

# Full table
print(f"\n  Complete inventory:")
print(f"  {'#':<4s} {'Column':<35s} {'Source':<6s} {'Category':<22s} Description")
print("  " + "-" * 100)
for i, row in inv.iterrows():
    print(f"  {i+1:<4d} {row['column']:<35s} {row['source']:<6s} "
          f"{row['category']:<22s} {row['description']}")

BLOCK 2: COMPLETE FACTOR INVENTORY

  Factors in data:       106
  Factors catalogued:    106
  ✓ Every factor in data is catalogued
  ✓ Every catalogued factor exists in data

  By source:
source
FRED    84
WRDS    22

  By category:
category
bond_returns        14
inflation           11
labor_market         8
consumer             8
surveys              6
commodities          5
orders               5
consumer_mom         5
housing              4
trade                4
inflation_yoy        4
tbill_returns        4
inflation_mom        4
housing_mom          4
money_supply         3
orders_mom           3
production           3
money_supply_mom     3
liquidity            2
trade_mom            2
commodities_mom      1
housing_yoy          1
labor_market_mom     1
production_mom       1

  ✓ Saved: ..\..\..\Data\Data_Collection\Final\Stage_1_5_Validation_and_Feature_Engineering\macro_monthly_descriptions_pre.csv
    106 factors

  Complete inventory:
  #    Column                        

In [5]:
# %% [markdown]
# ## Block 3: Feature Engineering
#
# Panel D already has extensive pre-computed features from the FRED cleaning
# stage (24 MoM changes, 5 YoY changes). Engineering here is deliberately
# minimal — only features that genuinely add new information.
#
# Three sections:
#   A. Economic acceleration (2nd derivative of key MoM — is the economy
#      speeding up or slowing down? Distinct from the level of growth.)
#   B. Bond term premium returns (long minus short bond returns — captures
#      duration risk compensation that matters for equity pricing)
#   C. Smoothed MoM (3-month rolling averages of volatile monthly series —
#      reduces noise in housing, payrolls, durables)
#
# No drops — all 106 factors are meaningful macro indicators.

# %%
print("=" * 90)
print("BLOCK 3: FEATURE ENGINEERING")
print("=" * 90)

n_before = df.shape[1]
new_features = []

df = df.sort_values('date').reset_index(drop=True)

# ═══════════════════════════════════════════════════════════════════════════════
# A. ECONOMIC ACCELERATION (2ND DERIVATIVE OF KEY MOM)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- A. Economic acceleration ---\n")
print("  Second derivative of MoM changes: is the rate of change increasing or")
print("  decreasing? Positive = economy accelerating, negative = decelerating.\n")

a_start = len(new_features)

# Key macro series where acceleration matters for equity markets
accel_pairs = [
    ('nonfarm_payrolls_mom', 'nfp_accel',            'Payroll growth acceleration'),
    ('cpi_core_mom',         'cpi_core_accel',        'Core CPI inflation acceleration'),
    ('pce_core_mom',         'pce_core_accel',        'Core PCE inflation acceleration'),
    ('retail_sales_mom',     'retail_accel',           'Retail sales growth acceleration'),
    ('indpro_mom',           'indpro_accel',           'Industrial production acceleration'),
    ('housing_starts_mom',   'housing_starts_accel',   'Housing starts growth acceleration'),
    ('durable_orders_mom',   'durable_orders_accel',   'Durable goods orders acceleration'),
    ('personal_spending_mom','spending_accel',          'Personal spending acceleration'),
    ('personal_income_mom',  'income_accel',            'Personal income acceleration'),
    ('exports_mom',          'exports_accel',           'Exports growth acceleration'),
    ('imports_mom',          'imports_accel',           'Imports growth acceleration'),
]

for src, name, desc in accel_pairs:
    if src in df.columns:
        df[name] = df[src].diff(1)
        new_features.append(name)

print(f"  Section A: {len(new_features) - a_start} acceleration features")
for f in new_features[a_start:]:
    v = df[f].dropna()
    print(f"    {f:<30s} median={v.median():.4f}  range=[{v.min():.4f}, {v.max():.4f}]")

# ═══════════════════════════════════════════════════════════════════════════════
# B. BOND TERM PREMIUM RETURNS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- B. Bond term premium returns ---\n")
print("  Long minus short bond returns: positive = long bonds outperformed")
print("  (rates fell or curve steepened). Negative = rates rose.\n")

b_start = len(new_features)

# 30y minus 2y return (long-duration vs short-duration)
if all(c in df.columns for c in ['b30ret', 'b2ret']):
    df['bond_30y_2y_spread_ret'] = df['b30ret'] - df['b2ret']
    new_features.append('bond_30y_2y_spread_ret')

# 10y minus T-bill return (benchmark term premium)
if all(c in df.columns for c in ['b10ret', 't90ret']):
    df['bond_10y_tbill_spread_ret'] = df['b10ret'] - df['t90ret']
    new_features.append('bond_10y_tbill_spread_ret')

# 30y minus 10y (long-end steepener/flattener)
if all(c in df.columns for c in ['b30ret', 'b10ret']):
    df['bond_30y_10y_spread_ret'] = df['b30ret'] - df['b10ret']
    new_features.append('bond_30y_10y_spread_ret')

# 5y minus 2y (belly of the curve)
if all(c in df.columns for c in ['b5ret', 'b2ret']):
    df['bond_5y_2y_spread_ret'] = df['b5ret'] - df['b2ret']
    new_features.append('bond_5y_2y_spread_ret')

# Cumulative bond returns (3-month bond momentum)
if 'b10ret' in df.columns:
    df['b10ret_cum_3m'] = df['b10ret'].rolling(3, min_periods=2).sum()
    new_features.append('b10ret_cum_3m')

if 'b30ret' in df.columns:
    df['b30ret_cum_3m'] = df['b30ret'].rolling(3, min_periods=2).sum()
    new_features.append('b30ret_cum_3m')

print(f"  Section B: {len(new_features) - b_start} bond term premium features")
for f in new_features[b_start:]:
    v = df[f].dropna()
    print(f"    {f:<35s} median={v.median():.4f}  range=[{v.min():.4f}, {v.max():.4f}]")

# ═══════════════════════════════════════════════════════════════════════════════
# C. SMOOTHED MOM (3-MONTH ROLLING AVERAGES)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- C. Smoothed MoM (3-month rolling averages) ---\n")
print("  Reduces noise in volatile monthly series. Gives a clearer picture")
print("  of the underlying trend vs single-month prints.\n")

c_start = len(new_features)

# Most volatile / noisy monthly series
smooth_pairs = [
    ('nonfarm_payrolls_mom', 'nfp_mom_3m',              'Smoothed payroll growth (3m avg)'),
    ('retail_sales_mom',     'retail_sales_mom_3m',      'Smoothed retail growth (3m avg)'),
    ('housing_starts_mom',   'housing_starts_mom_3m',    'Smoothed housing starts growth (3m avg)'),
    ('durable_orders_mom',   'durable_orders_mom_3m',    'Smoothed durable orders growth (3m avg)'),
    ('indpro_mom',           'indpro_mom_3m',            'Smoothed industrial production growth (3m avg)'),
    ('personal_spending_mom','spending_mom_3m',           'Smoothed personal spending growth (3m avg)'),
    ('new_home_sales_mom',   'new_home_sales_mom_3m',    'Smoothed new home sales growth (3m avg)'),
    ('cpi_core_mom',         'cpi_core_mom_3m',          'Smoothed core CPI MoM (3m avg)'),
    ('factory_orders_mom',   'factory_orders_mom_3m',    'Smoothed factory orders growth (3m avg)'),
    ('exports_mom',          'exports_mom_3m',           'Smoothed exports growth (3m avg)'),
]

for src, name, desc in smooth_pairs:
    if src in df.columns:
        df[name] = df[src].rolling(3, min_periods=2).mean()
        new_features.append(name)

print(f"  Section C: {len(new_features) - c_start} smoothed features")
for f in new_features[c_start:]:
    v = df[f].dropna()
    print(f"    {f:<35s} median={v.median():.4f}  range=[{v.min():.4f}, {v.max():.4f}]")

# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("BLOCK 3: SUMMARY")
print("=" * 90)

remaining = [c for c in df.columns if c != 'date']

print(f"\n  New features by section:")
print(f"    A. Economic acceleration:       {sum(1 for f in new_features if new_features.index(f) < b_start)}")
print(f"    B. Bond term premium:           {sum(1 for f in new_features if b_start <= new_features.index(f) < c_start)}")
print(f"    C. Smoothed MoM:                {sum(1 for f in new_features if new_features.index(f) >= c_start)}")
print(f"    ──────────────────────────────")
print(f"    Total new features:             {len(new_features)}")
print(f"\n  Columns dropped:                  0")
print(f"  Net column change:                {n_before} → {df.shape[1]} columns")
print(f"  Surviving factors:                {len(remaining)}")

# Verify
assert 'date' in df.columns
print(f"\n  ✓ Date column intact")

missing_new = [f for f in new_features if f not in df.columns]
if missing_new:
    print(f"  ✗ Missing: {missing_new}")
else:
    print(f"  ✓ All {len(new_features)} new features present")

# NaN check (1 warmup row for diff, 1-2 for rolling 3)
total_nan = df[remaining].isna().sum().sum()
warmup_rows = 3
warmup_nan = df.head(warmup_rows)[remaining].isna().sum().sum()
post_warmup_nan = df.iloc[warmup_rows:][remaining].isna().sum().sum()
print(f"\n  Total NaN: {total_nan:,}")
print(f"    In first {warmup_rows} rows (warmup): {warmup_nan:,}")
print(f"    After warmup: {post_warmup_nan:,}")

if post_warmup_nan == 0:
    print(f"  ✓ Zero NaN after warmup period")
else:
    post_warmup = df.iloc[warmup_rows:]
    nan_cols = post_warmup[remaining].isna().sum()
    nan_cols = nan_cols[nan_cols > 0].sort_values(ascending=False)
    print(f"  Columns with NaN after row {warmup_rows} "
          f"(expected: avg_hourly_earnings to May 2006:")
    for c in nan_cols.head(10).index:
        print(f"    {c}: {int(nan_cols[c])} NaN")

# Complete feature list
print(f"\n  Complete list of {len(new_features)} new features:")
for i, f in enumerate(new_features, 1):
    print(f"    {i:>3d}. {f}")

BLOCK 3: FEATURE ENGINEERING

--- A. Economic acceleration ---

  Second derivative of MoM changes: is the rate of change increasing or
  decreasing? Positive = economy accelerating, negative = decelerating.

  Section A: 11 acceleration features
    nfp_accel                      median=-0.0130  range=[-13.1499, 15.0507]
    cpi_core_accel                 median=-0.0004  range=[-0.5498, 0.5781]
    pce_core_accel                 median=0.0052  range=[-8.5947, 8.5590]
    retail_accel                   median=0.0240  range=[-12.2175, 36.5792]
    indpro_accel                   median=0.0454  range=[-11.8699, 11.5619]
    housing_starts_accel           median=-1.4612  range=[-43.9725, 40.3800]
    durable_orders_accel           median=0.3108  range=[-43.3170, 34.6596]
    spending_accel                 median=0.0125  range=[-7.6049, 22.5972]
    income_accel                   median=0.0176  range=[-33.8130, 28.3990]
    exports_accel                  median=-0.2377  range=[-18.2971, 20.

In [6]:
# %% [markdown]
# ## Block 4: Final Factor Inventory & Save

# %%
print("=" * 90)
print("BLOCK 4: FINAL FACTOR INVENTORY & SAVE")
print("=" * 90)

from pathlib import Path

OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD FINAL INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
final_inventory = []

def add(col, source, category, description):
    final_inventory.append({
        'column': col,
        'source': source,
        'category': category,
        'description': description,
    })

# ─────────────────────────────────────────────────────────────────────────────
# LABOR MARKET (8)
# ─────────────────────────────────────────────────────────────────────────────

add('nonfarm_payrolls',    'FRED', 'labor_market', 'Total nonfarm payrolls (thousands)')
add('unrate',              'FRED', 'labor_market', 'Unemployment rate (%)')
add('u6_rate',             'FRED', 'labor_market', 'U-6 underemployment rate (%)')
add('participation',       'FRED', 'labor_market', 'Labor force participation rate (%)')
add('avg_hourly_earnings', 'FRED', 'labor_market', 'Average hourly earnings, private ($)')
add('avg_weekly_hours',    'FRED', 'labor_market', 'Average weekly hours, private')
add('jolts_openings',      'FRED', 'labor_market', 'JOLTS job openings (thousands)')
add('jolts_quits',         'FRED', 'labor_market', 'JOLTS quits rate (%)')

# ─────────────────────────────────────────────────────────────────────────────
# INFLATION & PRICES (9)
# ─────────────────────────────────────────────────────────────────────────────

add('cpi_urban',     'FRED', 'inflation', 'CPI-U all items (index)')
add('cpi_core',      'FRED', 'inflation', 'CPI-U less food & energy (index)')
add('cpi_food',      'FRED', 'inflation', 'CPI food (index)')
add('cpi_energy',    'FRED', 'inflation', 'CPI energy (index)')
add('cpi_shelter',   'FRED', 'inflation', 'CPI shelter (index)')
add('cpi_services',  'FRED', 'inflation', 'CPI services (index)')
add('pce_price',     'FRED', 'inflation', 'PCE price index')
add('pce_core',      'FRED', 'inflation', 'Core PCE price index')
add('import_prices', 'FRED', 'inflation', 'Import price index')

# ─────────────────────────────────────────────────────────────────────────────
# PRODUCTION (3)
# ─────────────────────────────────────────────────────────────────────────────

add('indpro',     'FRED', 'production', 'Industrial production index')
add('cap_util',   'FRED', 'production', 'Capacity utilisation (%)')
add('manuf_prod', 'FRED', 'production', 'Manufacturing production index')

# ─────────────────────────────────────────────────────────────────────────────
# ORDERS & INVENTORIES (5)
# ─────────────────────────────────────────────────────────────────────────────

add('durable_orders',       'FRED', 'orders', 'Durable goods orders ($M)')
add('durable_ex_transport', 'FRED', 'orders', 'Durables ex-transport ($M)')
add('factory_orders',       'FRED', 'orders', 'Factory orders ($M)')
add('business_inventories', 'FRED', 'orders', 'Business inventories ($M)')
add('inv_sales_ratio',      'FRED', 'orders', 'Inventory-to-sales ratio')

# ─────────────────────────────────────────────────────────────────────────────
# CONSUMER (8)
# ─────────────────────────────────────────────────────────────────────────────

add('retail_sales',      'FRED', 'consumer', 'Retail sales ($M)')
add('retail_ex_auto',    'FRED', 'consumer', 'Retail sales ex-auto ($M)')
add('personal_income',   'FRED', 'consumer', 'Personal income ($B)')
add('personal_spending', 'FRED', 'consumer', 'Personal consumption ($B)')
add('saving_rate',       'FRED', 'consumer', 'Personal saving rate (%)')
add('consumer_credit',   'FRED', 'consumer', 'Consumer credit ($B)')
add('umich_sentiment',   'FRED', 'consumer', 'Michigan consumer sentiment')
add('vehicle_sales',     'FRED', 'consumer', 'Vehicle sales (M units, SAAR)')

# ─────────────────────────────────────────────────────────────────────────────
# HOUSING (4)
# ─────────────────────────────────────────────────────────────────────────────

add('housing_starts',   'FRED', 'housing', 'Housing starts (K units, SAAR)')
add('building_permits', 'FRED', 'housing', 'Building permits (K units, SAAR)')
add('new_home_sales',   'FRED', 'housing', 'New home sales (K, SAAR)')
add('case_shiller',     'FRED', 'housing', 'Case-Shiller national home price index')

# ─────────────────────────────────────────────────────────────────────────────
# TRADE (4)
# ─────────────────────────────────────────────────────────────────────────────

add('trade_balance',         'FRED', 'trade', 'Trade balance ($M, negative = deficit)')
add('exports',               'FRED', 'trade', 'Total exports ($M)')
add('imports',               'FRED', 'trade', 'Total imports ($M)')
add('trade_balance_12m_avg', 'FRED', 'trade', 'Trade balance 12m rolling avg ($M)')

# ─────────────────────────────────────────────────────────────────────────────
# MONEY SUPPLY (3)
# ─────────────────────────────────────────────────────────────────────────────

add('m1',            'FRED', 'money_supply', 'M1 ($B)')
add('m2',            'FRED', 'money_supply', 'M2 ($B)')
add('monetary_base', 'FRED', 'money_supply', 'Monetary base ($B)')

# ─────────────────────────────────────────────────────────────────────────────
# COMMODITIES (5)
# ─────────────────────────────────────────────────────────────────────────────

add('copper_monthly',   'FRED', 'commodities', 'Copper monthly avg ($/metric ton)')
add('aluminum_monthly', 'FRED', 'commodities', 'Aluminum monthly avg ($/metric ton)')
add('wheat_monthly',    'FRED', 'commodities', 'Wheat monthly avg ($/bushel)')
add('corn_monthly',     'FRED', 'commodities', 'Corn monthly avg ($/bushel)')
add('lumber_monthly',   'FRED', 'commodities', 'Lumber monthly avg ($/1000 board ft)')

# ─────────────────────────────────────────────────────────────────────────────
# SURVEYS (6)
# ─────────────────────────────────────────────────────────────────────────────

add('ism_manuf',       'FRED', 'surveys', 'ISM Manufacturing PMI')
add('ism_new_orders',  'FRED', 'surveys', 'ISM new orders')
add('philly_fed',      'FRED', 'surveys', 'Philly Fed Manufacturing')
add('empire_state',    'FRED', 'surveys', 'Empire State Manufacturing')
add('kansas_fed',      'FRED', 'surveys', 'Kansas City Fed Manufacturing')
add('chicago_fed_nai', 'FRED', 'surveys', 'Chicago Fed National Activity Index')

# ─────────────────────────────────────────────────────────────────────────────
# MOM CHANGES (24)
# ─────────────────────────────────────────────────────────────────────────────

add('nonfarm_payrolls_mom',     'FRED', 'labor_market_mom',  'Nonfarm payrolls MoM %')
add('cpi_urban_mom',            'FRED', 'inflation_mom',     'CPI-U MoM %')
add('cpi_core_mom',             'FRED', 'inflation_mom',     'Core CPI MoM %')
add('pce_price_mom',            'FRED', 'inflation_mom',     'PCE price MoM %')
add('pce_core_mom',             'FRED', 'inflation_mom',     'Core PCE MoM %')
add('indpro_mom',               'FRED', 'production_mom',    'Industrial production MoM %')
add('retail_sales_mom',         'FRED', 'consumer_mom',      'Retail sales MoM %')
add('retail_ex_auto_mom',       'FRED', 'consumer_mom',      'Retail ex-auto MoM %')
add('personal_income_mom',      'FRED', 'consumer_mom',      'Personal income MoM %')
add('personal_spending_mom',    'FRED', 'consumer_mom',      'Personal spending MoM %')
add('housing_starts_mom',       'FRED', 'housing_mom',       'Housing starts MoM %')
add('building_permits_mom',     'FRED', 'housing_mom',       'Building permits MoM %')
add('new_home_sales_mom',       'FRED', 'housing_mom',       'New home sales MoM %')
add('durable_orders_mom',       'FRED', 'orders_mom',        'Durable orders MoM %')
add('durable_ex_transport_mom', 'FRED', 'orders_mom',        'Durables ex-transport MoM %')
add('factory_orders_mom',       'FRED', 'orders_mom',        'Factory orders MoM %')
add('exports_mom',              'FRED', 'trade_mom',         'Exports MoM %')
add('imports_mom',              'FRED', 'trade_mom',         'Imports MoM %')
add('m1_mom',                   'FRED', 'money_supply_mom',  'M1 MoM %')
add('m2_mom',                   'FRED', 'money_supply_mom',  'M2 MoM %')
add('monetary_base_mom',        'FRED', 'money_supply_mom',  'Monetary base MoM %')
add('copper_monthly_mom',       'FRED', 'commodities_mom',   'Copper MoM %')
add('case_shiller_mom',         'FRED', 'housing_mom',       'Case-Shiller MoM %')
add('consumer_credit_mom',      'FRED', 'consumer_mom',      'Consumer credit MoM %')

# ─────────────────────────────────────────────────────────────────────────────
# YOY CHANGES (5)
# ─────────────────────────────────────────────────────────────────────────────

add('cpi_urban_yoy',    'FRED', 'inflation_yoy', 'CPI-U YoY % (headline)')
add('cpi_core_yoy',     'FRED', 'inflation_yoy', 'Core CPI YoY %')
add('pce_price_yoy',    'FRED', 'inflation_yoy', 'PCE price YoY %')
add('pce_core_yoy',     'FRED', 'inflation_yoy', 'Core PCE YoY % (Fed preferred)')
add('case_shiller_yoy', 'FRED', 'housing_yoy',   'Case-Shiller YoY %')

# ─────────────────────────────────────────────────────────────────────────────
# WRDS BOND RETURNS & INDICES (14)
# ─────────────────────────────────────────────────────────────────────────────

add('b30ret', 'WRDS', 'bond_returns', '30y Treasury monthly return')
add('b30ind', 'WRDS', 'bond_returns', '30y Treasury total return index')
add('b20ret', 'WRDS', 'bond_returns', '20y Treasury monthly return')
add('b20ind', 'WRDS', 'bond_returns', '20y Treasury total return index')
add('b10ret', 'WRDS', 'bond_returns', '10y Treasury monthly return')
add('b10ind', 'WRDS', 'bond_returns', '10y Treasury total return index')
add('b7ret',  'WRDS', 'bond_returns', '7y Treasury monthly return')
add('b7ind',  'WRDS', 'bond_returns', '7y Treasury total return index')
add('b5ret',  'WRDS', 'bond_returns', '5y Treasury monthly return')
add('b5ind',  'WRDS', 'bond_returns', '5y Treasury total return index')
add('b2ret',  'WRDS', 'bond_returns', '2y Treasury monthly return')
add('b2ind',  'WRDS', 'bond_returns', '2y Treasury total return index')
add('b1ret',  'WRDS', 'bond_returns', '1y Treasury monthly return')
add('b1ind',  'WRDS', 'bond_returns', '1y Treasury total return index')

# ─────────────────────────────────────────────────────────────────────────────
# WRDS T-BILL (4)
# ─────────────────────────────────────────────────────────────────────────────

add('t90ret', 'WRDS', 'tbill_returns', '90d T-bill monthly return')
add('t90ind', 'WRDS', 'tbill_returns', '90d T-bill total return index')
add('t30ret', 'WRDS', 'tbill_returns', '30d T-bill monthly return')
add('t30ind', 'WRDS', 'tbill_returns', '30d T-bill total return index')

# ─────────────────────────────────────────────────────────────────────────────
# WRDS LIQUIDITY & CPI (4)
# ─────────────────────────────────────────────────────────────────────────────

add('ps_level', 'WRDS', 'liquidity',  'Pastor-Stambaugh liquidity level')
add('ps_innov', 'WRDS', 'liquidity',  'Pastor-Stambaugh liquidity innovation')
add('cpiret',   'WRDS', 'inflation',  'Monthly CPI return (WRDS)')
add('cpiind',   'WRDS', 'inflation',  'CPI index (WRDS)')

# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 3 NEW: ECONOMIC ACCELERATION (Section A)
# ─────────────────────────────────────────────────────────────────────────────

add('nfp_accel',              'Derived', 'acceleration', 'Payroll growth acceleration (2nd derivative)')
add('cpi_core_accel',         'Derived', 'acceleration', 'Core CPI inflation acceleration')
add('pce_core_accel',         'Derived', 'acceleration', 'Core PCE inflation acceleration')
add('retail_accel',           'Derived', 'acceleration', 'Retail sales growth acceleration')
add('indpro_accel',           'Derived', 'acceleration', 'Industrial production acceleration')
add('housing_starts_accel',   'Derived', 'acceleration', 'Housing starts acceleration')
add('durable_orders_accel',   'Derived', 'acceleration', 'Durable orders acceleration')
add('spending_accel',         'Derived', 'acceleration', 'Personal spending acceleration')
add('income_accel',           'Derived', 'acceleration', 'Personal income acceleration')
add('exports_accel',          'Derived', 'acceleration', 'Exports growth acceleration')
add('imports_accel',          'Derived', 'acceleration', 'Imports growth acceleration')

# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 3 NEW: BOND TERM PREMIUM (Section B)
# ─────────────────────────────────────────────────────────────────────────────

add('bond_30y_2y_spread_ret',    'Derived', 'bond_term_premium', '30y-2y bond return spread')
add('bond_10y_tbill_spread_ret', 'Derived', 'bond_term_premium', '10y-Tbill bond return spread')
add('bond_30y_10y_spread_ret',   'Derived', 'bond_term_premium', '30y-10y bond return spread (long-end)')
add('bond_5y_2y_spread_ret',     'Derived', 'bond_term_premium', '5y-2y bond return spread (belly)')
add('b10ret_cum_3m',             'Derived', 'bond_momentum',     '10y bond cumulative 3m return')
add('b30ret_cum_3m',             'Derived', 'bond_momentum',     '30y bond cumulative 3m return')

# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 3 NEW: SMOOTHED MOM (Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('nfp_mom_3m',              'Derived', 'smoothed', 'Payroll MoM 3m rolling avg')
add('retail_sales_mom_3m',     'Derived', 'smoothed', 'Retail sales MoM 3m avg')
add('housing_starts_mom_3m',   'Derived', 'smoothed', 'Housing starts MoM 3m avg')
add('durable_orders_mom_3m',   'Derived', 'smoothed', 'Durable orders MoM 3m avg')
add('indpro_mom_3m',           'Derived', 'smoothed', 'Industrial production MoM 3m avg')
add('spending_mom_3m',         'Derived', 'smoothed', 'Spending MoM 3m avg')
add('new_home_sales_mom_3m',   'Derived', 'smoothed', 'New home sales MoM 3m avg')
add('cpi_core_mom_3m',         'Derived', 'smoothed', 'Core CPI MoM 3m avg')
add('factory_orders_mom_3m',   'Derived', 'smoothed', 'Factory orders MoM 3m avg')
add('exports_mom_3m',          'Derived', 'smoothed', 'Exports MoM 3m avg')

# ═══════════════════════════════════════════════════════════════════════════════
# VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
inv = pd.DataFrame(final_inventory)

remaining_factors = [c for c in df.columns if c != 'date']
catalogued = set(inv['column'])

in_data_not_catalogued = [c for c in remaining_factors if c not in catalogued]
in_catalogue_not_data = [c for c in catalogued if c not in remaining_factors]

print(f"\n  Factors in data:       {len(remaining_factors)}")
print(f"  Factors catalogued:    {len(inv)}")

if in_data_not_catalogued:
    print(f"\n  ✗ IN DATA but NOT catalogued ({len(in_data_not_catalogued)}):")
    for c in in_data_not_catalogued:
        print(f"    {c}")
else:
    print(f"  ✓ Every factor in data is catalogued")

if in_catalogue_not_data:
    print(f"\n  ✗ In catalogue but NOT in data ({len(in_catalogue_not_data)}):")
    for c in in_catalogue_not_data:
        print(f"    {c}")
else:
    print(f"  ✓ Every catalogued factor exists in data")

print(f"\n  By source:")
print(inv['source'].value_counts().to_string())

print(f"\n  By category:")
print(inv['category'].value_counts().to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE INVENTORY CSV
# ═══════════════════════════════════════════════════════════════════════════════

# %%
csv_path = OUT_DIR / 'macro_monthly_factor_inventory_final.csv'
csv_string = inv.to_csv(index=False)
with open(csv_path, 'w', encoding='utf-8') as f:
    f.write(csv_string)
print(f"\n  ✓ Inventory saved: {csv_path}")
print(f"    {len(inv)} factors")

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE ENGINEERED PANEL TO PARQUET
# ═══════════════════════════════════════════════════════════════════════════════

# %%
df = df.sort_values('date').reset_index(drop=True)

parquet_path = OUT_DIR / 'panel_macro_monthly_engineered.parquet'
df.to_parquet(parquet_path, index=False, engine='pyarrow')

file_size = parquet_path.stat().st_size
print(f"\n  ✓ Panel saved: {parquet_path}")
print(f"    {len(df):,} rows × {df.shape[1]} columns")
print(f"    Key: date")
print(f"    Factors: {len(remaining_factors)}")
print(f"    Size: {file_size / 1e3:.1f} KB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("PANEL D FEATURE ENGINEERING COMPLETE")
print("=" * 90)

print(f"""
  Pipeline: Raw (107 cols) → Block 1 (107) → Block 3 ({df.shape[1]}) → Final ({df.shape[1]})

  Final panel:
    Rows:       {len(df):,}
    Columns:    {df.shape[1]}
    Factors:    {len(remaining_factors)}
    Date range: {df['date'].min().date()} → {df['date'].max().date()}

  Saved to:
    Panel:     {parquet_path}
    Inventory: {csv_path}

  ═══════════════════════════════════════════════════════════════════════
  ALL FOUR PANELS COMPLETE — STAGE 1.5 FINISHED
  ═══════════════════════════════════════════════════════════════════════

  Panel A (stock daily):    525,957 rows × 192 factors  → panel_stock_daily_engineered.parquet
  Panel B (stock monthly):   25,194 rows × 197 factors  → panel_stock_monthly_engineered.parquet
  Panel C (macro daily):      4,656 rows × 209 factors  → panel_macro_daily_engineered.parquet
  Panel D (macro monthly):      222 rows × 133 factors  → panel_macro_monthly_engineered.parquet

  Total unique factors: ~731 (before cross-panel dedup in Step 3)

  Next step: Stage 2 — Aggregation
    Stock panels (A, B) get cap-weighted cross-sectional statistics
    (cwmean, cwstd, cwskew, cwkurt, spread) per date.
    Macro panels (C, D) pass through directly.
""")

BLOCK 4: FINAL FACTOR INVENTORY & SAVE

  Factors in data:       133
  Factors catalogued:    133
  ✓ Every factor in data is catalogued
  ✓ Every catalogued factor exists in data

  By source:
source
FRED       84
Derived    27
WRDS       22

  By category:
category
bond_returns         14
inflation            11
acceleration         11
smoothed             10
labor_market          8
consumer              8
surveys               6
commodities           5
orders                5
consumer_mom          5
housing               4
trade                 4
bond_term_premium     4
tbill_returns         4
inflation_yoy         4
housing_mom           4
inflation_mom         4
orders_mom            3
money_supply_mom      3
production            3
money_supply          3
liquidity             2
bond_momentum         2
trade_mom             2
commodities_mom       1
production_mom        1
housing_yoy           1
labor_market_mom      1

  ✓ Inventory saved: ..\..\..\Data\Data_Collection\Final\St